ゴールデンプロンプト
以下のテキストをコピーして使用してください。

Markdown
# Role
あなたは分散システムとMLOps構築に精通したシニアソフトウェアエンジニアです。
FastAPI, Redis, Celeryを使用し、LLM（Ollama）と音声認識（Whisper）を統合した「高負荷対応・分散タスクキューシステム」を構築してください。

# System Requirement
1. **マイクロサービス構成**:
   - 推論サーバ（Ollama, Whisper）とタスクワーカー（Celery）は分離する。
   - アプリケーションコードは「ドメイン駆動」でディレクトリを分離する（TranslationとTranscription）。
   - ビルド環境は「コンテナ最適化」のため、Dockerfileとrequirements.txtを各役割ごとに分離する。

2. **Celery Configuration (重要)**:
   - **Queue**: 'translation' と 'transcription' の2つを定義。
   - **Concurrency**: 重いAI処理のため、1ワーカープロセスにつき「同時実行数 1 (Strict)」とする。
   - **Reliability**: `task_acks_late = True`, `worker_prefetch_multiplier = 1` を設定し、タスクの抱え込みとロストを防ぐ。

3. **Workers Architecture**:
   - Worker A: 翻訳専用 ('translation' queueのみ)
   - Worker B: 文字起こし専用 ('transcription' queueのみ)
   - Worker C & D: 兼用 (リソースが空いている時に両方のqueueを処理)

4. **Directory Structure**:
   以下の構造を厳守してコードを生成すること。
```
text
   .
   ├── app/
   │   ├── __init__.py
   │   ├── config.py             # Celery設定 (include=['app.translation.tasks', ...])
   │   ├── main.py               # FastAPI entrypoint (Router統合)
   │   ├── translation/          # [翻訳ドメイン]
   │   │   ├── __init__.py
   │   │   ├── router.py         # POST /translate
   │   │   └── tasks.py          # Ollama API Client
   │   └── transcription/        # [文字起こしドメイン]
   │       ├── __init__.py
   │       ├── router.py         # POST /transcribe (ファイル保存処理含む)
   │       └── tasks.py          # Whisper API Client (/data 内のファイルパス処理)
   ├── data/                     # NFS共有用 (空で作成)
   ├── docker/                   # [Docker定義]
   │   ├── api.Dockerfile
   │   ├── translation.Dockerfile
   │   ├── transcription.Dockerfile
   │   └── general.Dockerfile
   ├── requirements/             # [依存関係]
   │   ├── common.txt            # Celery, Redis等
   │   ├── api.txt               # FastAPI等
   │   ├── translation.txt       # 翻訳用
   │   └── transcription.txt     # 音声処理用
   └── docker-compose.yml
```

# Implementation Details
API Logic:

 - /translate: テキストを受け取りOllama APIへ。

 - /transcribe: 音声ファイルを受け取り、共有ボリューム /data に保存した後、パスをWhisper APIへ渡す。

Docker Compose:

build context はルート(.)を指定し、dockerfile パスで各docker/配下のファイルを指定すること。

OllamaとWhisperのサービス定義を含める（Whisperは簡易的に python -m http.server 等のダミーでも可だが、接続設定は記述すること）。

# Output
上記のディレクトリ構造に従い、以下のファイルの全コードを生成してください。

 - requirements/ 配下の全ファイル

 - docker/ 配下の全Dockerfile

 - docker-compose.yml

 - app/config.py, app/main.py

 - app/translation/ 配下のコード

 - app/transcription/ 配下のコード


---

### プロンプトの解説（エンジニア向け）

このプロンプトには、意図通りの出力を得るための「トリガー」が仕込まれています。

1.  **Directory Structureの図示**:
    * AIにファイルパスを誤認させないため、ツリー構造を明示しました。これにより、`app/translation/tasks.py` といった深い階層のファイルも正確に生成されます。
2.  **`build context` の指定**:
    * Docker分離構成で最もハマりやすい「ContextはルートだがDockerfileはサブディレクトリ」という設定を、`Implementation Details` で明示的に指示しています。
3.  **Configの `include` 指定**:
    * `app/config.py` 内で `include=['app.translation.tasks', ...]` と書かせないと、Celeryがタスクを認識できない問題を未然に防いでいます。
4.  **各ワーカーの役割定義**:
    * Worker A/B/C/D のキュー割り当て（`-Q` オプション）を `docker-compose.yml` に正しく反映させるための指示です。

#.githubignore
```
#プロジェクト固有

models/
data/


# Jupyter Notebookのチェックポイント（自動保存ファイル）
.ipynb_checkpoints/

# IPykernelの接続ファイル
~/.ipython/

# 出力ログや一時ファイル
*.log

# 仮想環境（Jupyter用に作成している場合）
.venv/
venv/
env/

# Pythonのキャッシュ
__pycache__/
*.py[cod]

# 設定ファイル（パスワードなどが含まれる場合があるため）
.env

# --- Python Standard ---
__pycache__/
*.py[cod]
*$py.class
.env
.venv/
env/
venv/
build/
dist/
*.egg-info/

# --- uv Specific ---
# uv が作成する一時的なキャッシュディレクトリ
.uv/

# uv python install でインストールされた Python インタープリタ（パスを指定している場合）
.python-version

# --- IDEs ---
.vscode/
.idea/

# --- Testing & Coverage ---
.pytest_cache/
.coverage
htmlcov/
```

git config --global -edit
```
[init]
        defaultBranch = main
[user]
        email = tsummerbiz@gmail.com
        name = tsummerbiz
```


In [ ]:
git remote add origin git@github.com:tsummerbiz/ollama-test.git
git branch -M main
git push -u origin main